<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook D01: Neural Network Foundations</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook D01: Neural Network Foundations](../notebooks/D01_Neural_networks_intro.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The features, the tensors and the training loop from the notebook. `train_network` takes one extra
argument here — `dropout`, which the notebook passes to `build_network` directly — so that both exercises
can vary it.

Each 300-epoch run takes a few seconds on CPU. Both exercises train five seeds per setting, because
Notebook D01 measured a test-MAE spread across seeds of roughly 30 MAE, and a single-seed comparison
cannot see past it.

In [ ]:
import sys
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    print(f"PyTorch {torch.__version__}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]

TEST_DAYS, VALIDATION_DAYS = 90, 90
test_start = len(X) - TEST_DAYS
validation_start = test_start - VALIDATION_DAYS

y_validation = y.iloc[validation_start:test_start]
y_test = y.iloc[test_start:]


def prepare_tensors(X, y, validation_start, test_start):
    """Standardise on the training period only, and convert to tensors."""
    feature_scaler, target_scaler = StandardScaler(), StandardScaler()

    X_train_raw = X.iloc[:validation_start]
    y_train_raw = y.iloc[:validation_start].values.reshape(-1, 1)
    feature_scaler.fit(X_train_raw)
    target_scaler.fit(y_train_raw)

    def to_tensor(values):
        return torch.tensor(np.asarray(values, dtype=np.float32))

    tensors = {
        "X_train": to_tensor(feature_scaler.transform(X_train_raw)),
        "y_train": to_tensor(target_scaler.transform(y_train_raw)),
        "X_validation": to_tensor(feature_scaler.transform(X.iloc[validation_start:test_start])),
        "X_test": to_tensor(feature_scaler.transform(X.iloc[test_start:])),
    }

    def invert(predictions):
        return target_scaler.inverse_transform(
            np.asarray(predictions).reshape(-1, 1)
        ).ravel()

    return tensors, invert


def build_network(n_features, hidden=(64, 32), dropout=0.1):
    layers = []
    previous = n_features

    for width in hidden:
        layers += [nn.Linear(previous, width), nn.ReLU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
        previous = width

    layers.append(nn.Linear(previous, 1))
    return nn.Sequential(*layers)


def train_network(tensors, invert, y_validation, seed=0, epochs=300, batch_size=32,
                  learning_rate=1e-3, weight_decay=1e-4, dropout=0.1):
    """As in the notebook, with `dropout` exposed as an argument."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    network = build_network(tensors["X_train"].shape[1], dropout=dropout)
    optimiser = torch.optim.Adam(
        network.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(tensors["X_train"], tensors["y_train"]),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    history = []
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        network.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(network(batch_X), batch_y).backward()
            optimiser.step()

        network.eval()
        with torch.no_grad():
            training_loss = loss_function(network(tensors["X_train"]), tensors["y_train"]).item()
            validation_prediction = invert(network(tensors["X_validation"]).numpy())

        validation_mae = mean_absolute_error(y_validation, validation_prediction)
        history.append({"epoch": epoch, "training_loss": training_loss,
                        "validation_mae": validation_mae})

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae, "epoch": epoch,
                    "weights": {k: v.clone() for k, v in network.state_dict().items()}}

    network.load_state_dict(best["weights"])
    network.eval()
    return network, pd.DataFrame(history), best


def test_mae(network, tensors, invert):
    with torch.no_grad():
        return mean_absolute_error(y_test, invert(network(tensors["X_test"]).numpy()))


if TORCH_AVAILABLE:
    tensors, to_original_units = prepare_tensors(X, y, validation_start, test_start)
    print(f"{X.shape[1]} features, train {validation_start} days")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Set `dropout=0.0` and `weight_decay=0.0` in `train_network` and plot the curves again. How much sooner does the validation error start rising, and what does that tell you about what those two settings were doing?

In [ ]:
SETTINGS = {
    "Regularised": dict(dropout=0.1, weight_decay=1e-4),
    "Unregularised": dict(dropout=0.0, weight_decay=0.0),
}
SEEDS = range(5)

runs, curves = [], {}

if TORCH_AVAILABLE:
    for label, settings in SETTINGS.items():
        for seed in SEEDS:
            network, history, best = train_network(
                tensors, to_original_units, y_validation, seed=seed, **settings
            )
            if seed == 0:
                curves[label] = history

            runs.append({
                "setting": label,
                "seed": seed,
                "best epoch": best["epoch"],
                "best validation MAE": best["validation_mae"],
                "validation MAE at epoch 300": history["validation_mae"].iloc[-1],
                "final training loss": history["training_loss"].iloc[-1],
                "test MAE": test_mae(network, tensors, to_original_units),
            })

    by_setting = pd.DataFrame(runs).groupby("setting").mean(numeric_only=True).drop(columns="seed")
    print("Averaged over 5 seeds\n")
    print(by_setting.round(3).to_string())

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    colours = {"Regularised": "steelblue", "Unregularised": "crimson"}

    for label, history in curves.items():
        axes[0].plot(history["epoch"], history["training_loss"],
                     color=colours[label], linewidth=1.2, label=label)
        axes[1].plot(history["epoch"], history["validation_mae"],
                     color=colours[label], linewidth=1.2, label=label)
        best_epoch = int(history["validation_mae"].idxmin())
        axes[1].axvline(best_epoch, color=colours[label], linestyle="--", linewidth=1.0)
        axes[1].annotate(f"epoch {best_epoch}", (best_epoch, history["validation_mae"].max()),
                         color=colours[label], fontsize=9, ha="center")

    axes[0].set_title("Training loss (scaled units)", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("MSE")
    axes[0].set_yscale("log")
    axes[1].set_title("Validation MAE (original units)", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("MAE")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**Nearly three times sooner: the validation error bottoms out at epoch 54 on average without
regularisation, against epoch 143 with it.**

The two panels show the same run from both sides, and it is worth reading them together rather than
separately.

**The training loss panel is the more damning one.** The unregularised network reaches a final training
MSE of 0.001, nine times lower than the regularised network's 0.009. It is fitting the training data
substantially better. And it is *worse* at the only thing that matters: 361 best validation MAE against
336, and 370 on test against 327.

That gap — better on the data it has seen, worse on the data it has not — is what overfitting means,
stated as two numbers rather than as a shape on a chart. The network has enough capacity to memorise 734
training days, and with nothing holding it back, that is what it spends its epochs doing.

**What the two settings were doing, then, is three distinct things**, which the table separates:

1. **Delaying the peak**, epoch 54 to 143. More epochs of genuine improvement before memorisation takes
   over.
2. **Lowering the floor**, 361 to 336. Not merely reaching the same place more slowly — reaching a better
   place. Dropout forces the network to spread its representation across units rather than routing each
   prediction through one specialised path, and that redundancy generalises.
3. **Containing the damage afterwards**, 482 to 390 at epoch 300. The unregularised network does not just
   stop improving; it degrades steeply.

There is a fourth effect that does not show up in the score at all, and it is the one worth carrying
into practice.

**Early stopping hid most of this.** Both runs restore the best-validation weights at the end, so the
notebook's reported test MAE is measured at epoch 54 for one and epoch 143 for the other — never at the
epoch-300 disaster. Without that, the unregularised network would be shipped at 482 validation MAE instead
of 361.

So the three mechanisms are doing overlapping jobs, and you should not think of them as alternatives:

- **Early stopping** is a safety net. It bounds the damage from training too long, and it costs one
  validation split.
- **Dropout and weight decay** change where the floor is. They make the best achievable model better, which
  early stopping cannot do.

The practical reading of the numbers is that **regularisation bought 25 MAE of floor and 89 epochs of
headroom**, and that the headroom is what lets a slightly-too-long training schedule survive contact with
reality. Notebook [D02](../notebooks/D02_Recurrent_networks.ipynb) trains larger models where both effects
are considerably stronger.

One measurement caution, since it applies to everything in Part D: the numbers above are means over five
seeds, and the seed-to-seed spread on test MAE is about 30 MAE for the regularised network. The 25-MAE
difference in *validation* floor is consistent across seeds; the 43-MAE difference in *test* MAE is not
something a single run would establish. Notebook D01's own seed table makes the same point, and it is the
reason this solution averages rather than reporting one run.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Retrain the MLP with only the calendar and known-in-advance features, dropping every lag and rolling column. How much worse does it get, and is the difference larger than the seed-to-seed spread measured above? The comparison is the value of the history the network cannot see for itself.

In [ ]:
calendar_only = [column for column in X.columns
                 if not column.startswith(("lag_", "roll_"))]

print(f"Dropped {X.shape[1] - len(calendar_only)} history columns, kept {len(calendar_only)}:")
print("   ", ", ".join(calendar_only))

if TORCH_AVAILABLE:
    calendar_tensors, calendar_invert = prepare_tensors(
        X[calendar_only], y, validation_start, test_start
    )

    calendar_runs = []
    for seed in SEEDS:
        network, _, best = train_network(
            calendar_tensors, calendar_invert, y_validation, seed=seed
        )
        calendar_runs.append({
            "seed": seed,
            "validation MAE": best["validation_mae"],
            "test MAE": test_mae(network, calendar_tensors, calendar_invert),
        })

    full_runs = pd.DataFrame(runs).query("setting == 'Regularised'")

    comparison = pd.DataFrame({
        "All 20 features": full_runs["test MAE"].agg(["mean", "std", "min", "max"]),
        "Calendar only (11)": pd.DataFrame(calendar_runs)["test MAE"].agg(
            ["mean", "std", "min", "max"]
        ),
    })

    print()
    print("Test MAE over 5 seeds")
    print(comparison.round(1).to_string())

**About 20 MAE worse — 347 against 327 — and no, that is well inside the seed-to-seed spread.** The
calendar-only model's five seeds range from 287 to 456; the full model's from 301 to 382. The ranges
overlap almost completely, and the calendar-only model's *best* seed beats every seed of the full model.

**On this series, nine lag and rolling columns are worth approximately nothing.** That is not the answer
the exercise expects, and it is the correct one.

It is also consistent with everything Part C measured, once you go back and look:

- Permutation importance in Notebook [C01](../notebooks/C01_Feature_engineering.ipynb) put `open` at 1278
  and `promo` at 207, with `lag_1` at 88 and every other lag and rolling column in single digits.
- The leakage exercise in C01 found that even *tomorrow's* sales barely moved the score, because
  consecutive open days at this store correlate at only 0.19.
- The best-performing prune in C01 kept sixteen features and dropped eight, most of them rolling columns.

All three say the same thing: **for this store, sales are a function of the calendar, not of recent
history.** Whether the shop is open, whether a promotion is running, and which day of the week it is
determine almost everything; what the shop took last Tuesday adds very little on top. A single store's
daily takings are noisy around a calendar-driven level, and noise does not carry forward.

**The practical consequence is larger than the 20 MAE suggests, and it runs in the opposite direction
to the score.**

A model built on lags can only forecast one step before it needs a value it does not have. To reach day 30
it must either predict its way there — feeding forecasts back in as inputs, accumulating error at every
step, the drift Notebook [C02](../notebooks/C02_Machine_learning_models.ipynb) ran into — or be retrained
as a separate direct model for each horizon.

A model built only on calendar and known-in-advance columns has no such limit. Every input for any future
date is available today: the calendar is arithmetic, and the store's opening hours and promotion schedule
are planned in advance. **It can forecast day 1 and day 300 in the same call, with the same accuracy, and
no recursion.**

So the comparison is not "20 MAE worse". It is "20 MAE worse at horizon 1, and unboundedly better at
horizon 30", and which of those matters depends entirely on what the forecast is for. Weekly staff
rostering needs seven days; ordering stock from a supplier with a six-week lead time needs six weeks.

Two qualifications before generalising any of this:

- **This is one store.** Aggregate the chain and the lags would matter considerably more, because
  averaging across stores removes the idiosyncratic daily noise and leaves the slower-moving level that
  history actually predicts. The same is true of the hourly electricity series in Notebook
  [D02](../notebooks/D02_Recurrent_networks.ipynb), where consecutive observations are strongly dependent
  and the lags carry most of the signal.
- **"Known in advance" is doing real work here, and it is a property of the data, not the model.** `open`
  and `promo` are genuinely known for future dates. If the promotion schedule were decided week by week,
  these columns would be forecasts themselves, and the clean-extrapolation argument above would collapse.
  Check that a known-in-advance column really is known before you rely on it — it is the same question as
  the leakage check in C01, asked about the future rather than the past.

---

Back to [Notebook D01](../notebooks/D01_Neural_networks_intro.ipynb), or on to
[Notebook D02](../notebooks/D02_Recurrent_networks.ipynb).